# 11차시 실습 — 내 MVP 문서로 '프로덕션 RAG' (청킹·검색·평가)

> **day1·10차시 연속 실습.** 10차시에서는 짧은 문장 5개를 임베딩으로 검색했습니다.
> 오늘은 같은 서비스를 이어받아, **긴 문서**를 다루는 **프로덕션 수준 RAG**로 키웁니다.
> **청킹 → 임베딩·인덱싱(numpy → 벡터 DB) → 검색 → 생성 → 골든셋 평가**, 그리고 **청크 크기 실험**까지.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 순서대로 실행하세요(`Shift+Enter`).

In [27]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai numpy
import os, json, getpass
import numpy as np
from openai import OpenAI

key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
key = key.strip()
if not key.isascii():
    bad = sorted({ch for ch in key if not ch.isascii()})
    raise ValueError(f"API 키에 유니코드 문자 {bad} 가 섞여 있습니다 — "
                     "복사 과정에서 하이픈(-)이 대시(—)로 바뀐 경우가 대부분입니다. 키를 다시 붙여넣으세요.")
os.environ["OPENAI_API_KEY"] = key
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

10차시: "환불은 14일 이내" 같은 **짧은 문장 5개**를 임베딩으로 검색했습니다 — 되는 데모.
11차시: 같은 서비스의 **긴 안내 문서**(한 문서에 여러 사실이 섞임)를 다룹니다.

> 긴 문서를 통째로 넣으면 비싸고 검색 정확도도 떨어집니다 → **청킹**이 필요합니다.
> 그리고 "잘 되는지"를 느낌이 아니라 **골든셋으로 측정**합니다.
> 구조는 동일하니 **STEP 19에서 내 팀 MVP 문서로 그대로 교체**하면 됩니다.

## STEP 1 — 긴 문서 + 청킹 (Ingestion ①)

긴 문서는 **작은 의미 단위 조각(chunk)** 으로 나눕니다. 한 청크에 한 가지 사실이 담기도록.
아래는 글자 수 기반의 간단한 청커입니다(겹침 overlap 포함). 실전에선 문장·섹션 경계를 존중하면 더 좋습니다.

⌨️ **터미널 Codex:** `codex "긴 한국어 안내 문서를 글자 수 기준으로 겹침을 두고 청킹하는 함수를 만들어줘"`

In [28]:
# 내 서비스의 '긴 안내 문서' (실제로는 MVP의 규정/FAQ/매뉴얼 자리)
# ⚠ 설계 포인트: 앞 4개 문서는 질문의 '주어'와 '정답 숫자'가 한 문장 안에서 60자 이상
#   떨어져 있고, 뒤 2개는 어휘만 겹치는 '방해 문서'다 — STEP 10 실험에서 효과가 드러난다.
DOCS = [
    ("환불정책",
     "환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접수할 수 있으며, "
     "상품 상태 확인과 구매 증빙(영수증 또는 결제 내역) 검토 절차를 모두 거친 뒤, "
     "구매일 기준 14일 안에 접수된 건에 한해 처리된다. "
     "단, 디지털 상품과 일부 할인 상품은 대상에서 제외된다. 승인 후에는 영업일 기준 3일 안에 결제가 취소된다."),
    ("근무정책",
     "정규 근무 시간은 오전 9시부터 오후 6시까지이며, 점심시간은 12시부터 1시까지다. "
     "재택근무는 주 2회까지 신청할 수 있고, 반드시 팀장의 사전 승인이 필요하다. "
     "초과 근무는 사전 신청한 경우에만 인정되며, 보상휴가로 대체할 수 있다."),
    ("식대지원",
     "점심 식대의 경우 정규 근무일에 출근한 임직원을 대상으로 하며, 법인카드 사용을 원칙으로 하되 "
     "부득이하면 개인 결제 후 증빙을 첨부해 정산할 수 있고, 한도는 1인당 하루 1만 2천원으로 정해져 있다. "
     "야근을 하는 경우에는 별도로 1만원이 추가로 지급된다. 주말·공휴일 근무분은 사전 승인된 경우에만 인정된다."),
    ("시설안내",
     "방문객 주차의 경우 지하 2층에 마련된 외부인 전용 구역을 이용하게 되며, "
     "1층 안내데스크에서 방문 등록을 마치고 차량 번호가 확인된 경우에 한해 최초 2시간까지는 요금이 부과되지 않는다. "
     "이후에는 30분당 1천원이 부과된다. 사무실 와이파이 비밀번호는 1층 안내데스크에서 받을 수 있다. "
     "회의실 예약은 사내 포털에서 하며, 최대 90분까지 연속 예약이 가능하다."),
    # ↓ 방해 문서: 위 문서들과 어휘(상품·영수증·정산·신청…)가 겹치지만 정답은 없다
    ("교환반품정책",
     "개봉하지 않은 상품은 수령일로부터 30일 이내에 교환을 신청할 수 있으며, "
     "단순 변심에 의한 반품은 왕복 배송비를 고객이 부담한다. "
     "상품 하자로 인한 교환은 배송비 없이 처리되며, 영수증 또는 결제 내역이 있어야 접수된다."),
    ("출장정책",
     "출장 중 사용한 경비는 복귀 후 5일 안에 정산 시스템에 등록해야 하며, 숙박비와 교통비는 영수증 첨부가 필수다. "
     "출장지에서의 저녁 비용은 별도 기준을 따르며, 국내 출장 일비는 하루 3만원이 지급된다."),
]

def chunk_text(text, size=120, overlap=30):
    """글자 수 기준으로 겹침을 두고 분할."""
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += size - overlap
    return chunks

def build_chunks(docs, size=120, overlap=30):
    """(출처, 청크텍스트) 목록으로 평탄화 = 메타데이터 함께 보존."""
    out = []
    for title, body in docs:
        for c in chunk_text(body, size, overlap):
            out.append((title, c))
    return out

chunks = build_chunks(DOCS, size=120, overlap=30)
print(f"문서 {len(DOCS)}개 → 청크 {len(chunks)}개")
for src, c in chunks[:3]:
    print(f"  [{src}] {c[:40]}...")

문서 6개 → 청크 14개
  [환불정책] 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접수할...
  [환불정책]  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디지털...
  [환불정책] 취소된다....


## STEP 2 — 임베딩 · 인덱싱 (Ingestion ②, numpy)

각 청크를 `text-embedding-3-large`로 **의미 벡터**로 바꿔 numpy 행렬에 저장합니다 = **인제스션(한 번만)**.
임베딩은 정규화된 **단위 벡터**라, 유사도는 **점곱(dot) = 코사인 유사도** 한 번으로 끝납니다.

⌨️ **터미널 Codex:** `codex "청크 리스트를 text-embedding-3-large로 임베딩해 numpy 행렬로 인덱싱해줘"`

In [29]:
def embed(texts):
    resp = client.embeddings.create(model="text-embedding-3-large", input=texts)
    return np.array([d.embedding for d in resp.data])

def build_index(chunks):
    """청크들을 임베딩해 (정규화된) 행렬로 = 벡터 인덱스."""
    vecs = embed([c for _, c in chunks])
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # 단위 벡터로 정규화
    return vecs

chunk_vecs = build_index(chunks)   # 인제스션: 한 번만
print("인덱스 완성:", chunk_vecs.shape, "(청크 수 × 임베딩 차원)")

인덱스 완성: (14, 3072) (청크 수 × 임베딩 차원)


## STEP 3 — 벡터 DB에 넣기 (Chroma) — Ingestion ③

numpy 행렬은 훌륭한 학습 도구지만, 실전에서는 **벡터 DB**가 저장·top-k 검색·메타데이터 필터를 대신해 줍니다 — 프로세스가 죽어도 인덱스가 남고, 문서가 수백만 개로 늘어도 빠릅니다.

| 제품 | 특징 |
|---|---|
| **Pinecone** | 완전 매니지드 SaaS — 서버리스·자동 스케일, 운영 부담 최소 |
| **Chroma** | 오픈소스·로컬 — `pip install`로 끝. **오늘 실습에서 사용** |
| **Weaviate / Qdrant** | 오픈소스 + 클라우드 — 하이브리드 검색·강력한 필터링 |
| **pgvector** | PostgreSQL 확장 — 이미 Postgres를 쓰면 추가 인프라 없이 도입 |

제품이 달라도 개념은 같습니다: **upsert(벡터+메타데이터) → query(top-k)**. 여기서는 로컬 Chroma로 같은 청크를 인덱싱합니다.

⌨️ **터미널 Codex:** `codex "청크들을 ChromaDB 컬렉션에 upsert하고 OpenAI 임베딩으로 top-k 검색하는 RAG를 만들어줘"`


In [30]:
# ✅ 벡터 DB(Chroma)로 인덱싱 — numpy 행렬 대신 DB 컬렉션에 upsert
%pip install -q chromadb
import chromadb

db = chromadb.Client()   # 로컬 인메모리 — 파일로 남기려면 chromadb.PersistentClient(path="./chroma")
try: db.delete_collection("mvp_docs")      # 재실행 대비: 지우고 새로 만든다
except Exception: pass
col = db.create_collection("mvp_docs", metadata={"hnsw:space": "cosine"})

col.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=[v.tolist() for v in chunk_vecs],          # STEP 2에서 만든 임베딩 재사용
    documents=[c for _, c in chunks],
    metadatas=[{"source": src} for src, _ in chunks],     # 출처 = 메타데이터
)
print("컬렉션 저장 완료:", col.count(), "청크")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
컬렉션 저장 완료: 14 청크


## STEP 4 — 검색 + 생성 (Query Flow = RAG)

질문을 같은 모델로 임베딩 → 모든 청크와 **코사인 top-k** → 그 청크**만** 근거로 답하고 **출처**를 함께 냅니다.
검색 청크만 컨텍스트로 쓰므로 모델이 **지어내지 않습니다(환각↓)**.

⌨️ **터미널 Codex:** `codex "코사인 top-k로 검색하고 검색된 청크만 근거로 출처와 함께 답하는 RAG 함수를 만들어줘"`

In [31]:
def search(query, vecs, chunks, k=3):
    qv = embed([query])[0]
    qv = qv / np.linalg.norm(qv)
    scores = vecs @ qv                       # 단위 벡터 점곱 = 코사인 유사도
    order = np.argsort(scores)[::-1][:k]
    return [(chunks[i][0], chunks[i][1], float(scores[i])) for i in order]

def answer(query, vecs, chunks, k=3, verbose=True):
    hits = search(query, vecs, chunks, k)
    context = "\n".join(f"- ({src}) {c}" for src, c, _ in hits)
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role":"system","content":"아래 <문서>의 내용만 근거로 한국어로 답하라. 문서에 없으면 '문서에 없습니다'라고 답하고, 답 끝에 근거 출처를 [출처]로 표기하라."},
        {"role":"user","content":f"<문서>\n{context}\n</문서>\n\n질문: {query}"},
    ])
    out = resp.choices[0].message.content
    if verbose:
        print("📚 검색된 청크:")
        for src, c, s in hits:
            print(f"  ({s:.3f}) [{src}] {c[:38]}...")
        print(f"\n🤖 답: {out}")
    return out, hits

answer("환불은 며칠 안에 해야 하고 영수증이 필요해?", chunk_vecs, chunks)

📚 검색된 청크:
  (0.666) [환불정책] 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접...
  (0.550) [환불정책]  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디...
  (0.517) [교환반품정책] 비 없이 처리되며, 영수증 또는 결제 내역이 있어야 접수된다....

🤖 답: 환불은 구매일 기준 14일 안에 신청해야 하며, 영수증 또는 결제 내역이 필요합니다. [출처]


('환불은 구매일 기준 14일 안에 신청해야 하며, 영수증 또는 결제 내역이 필요합니다. [출처]',
 [('환불정책',
   '환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접수할 수 있으며, 상품 상태 확인과 구매 증빙(영수증 또는 결제 내역) 검토 절차를 모두 거친 뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리',
   0.6658366519081604),
  ('환불정책',
   ' 뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디지털 상품과 일부 할인 상품은 대상에서 제외된다. 승인 후에는 영업일 기준 3일 안에 결제가 취소된다.',
   0.5500759822732106),
  ('교환반품정책', '비 없이 처리되며, 영수증 또는 결제 내역이 있어야 접수된다.', 0.5170916413138256)])

## STEP 5 — 같은 RAG를 벡터 DB로 (Query via Chroma)

STEP 4의 numpy 검색을 **Chroma 질의**로 바꿔 봅니다. 개념은 동일(질의 임베딩 → top-k)이고, DB가 주는 보너스:

- **영속성** — 인덱스가 프로세스 밖에 남는다 (`PersistentClient`면 파일로)
- **메타데이터 필터** — `where={"source": "환불정책"}` 처럼 조건을 걸고 검색

⌨️ **터미널 Codex:** `codex "ChromaDB에서 top-k 검색하고 where 메타데이터 필터로 특정 문서 안에서만 검색하는 RAG를 만들어줘"`

In [32]:
# ✅ 벡터 DB로 검색 → 생성 — numpy search()/answer()와 같은 개념, 실행 주체만 DB
def search_db(query, k=3, where=None):
    qv = embed([query])[0]
    qv = (qv / np.linalg.norm(qv)).tolist()
    res = col.query(query_embeddings=[qv], n_results=k, where=where)
    return [(m["source"], d, 1 - dist)                    # cosine distance → 유사도
            for m, d, dist in zip(res["metadatas"][0], res["documents"][0], res["distances"][0])]

def answer_db(query, k=3, where=None):
    hits = search_db(query, k, where)
    context = "\n".join(f"- ({src}) {c}" for src, c, _ in hits)
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role":"system","content":"아래 <문서>의 내용만 근거로 한국어로 답하라. 문서에 없으면 '문서에 없습니다'라고 답하고, 답 끝에 근거 출처를 표기하라."},
        {"role":"user","content":f"<문서>\n{context}\n</문서>\n\n질문: {query}"}])
    return resp.choices[0].message.content

print("[전체 검색]", answer_db("환불은 며칠 이내에 가능해?"))
print()
# 메타데이터 필터: '시설안내' 문서 안에서만 검색 → 환불 정보가 없으니 모른다고 답해야 정상
print("[필터: source=시설안내]", answer_db("환불은 며칠 이내에 가능해?", where={"source": "시설안내"}))

[전체 검색] 환불은 구매일 기준 14일 안에 접수된 건에 한해 가능하다. (근거 출처: 문서)

[필터: source=시설안내] 문서에 없습니다.


## STEP 6 — 골든셋으로 평가 (검색 품질 + 답변 품질)

"느낌"이 아니라 **측정**합니다. 골든셋 = `질문 + 기대 근거(검색에 떠야 할 청크의 출처/키워드) + 정답 키워드`.
- **검색 품질(recall@k)**: 기대 근거가 top-k 안에 들어왔나
- **답변 품질**: 생성된 답에 정답 키워드가 들어있나

⌨️ **터미널 Codex:** `codex "골든셋으로 검색 recall@k와 답변 정확도를 자동 채점하는 함수를 만들어줘"`

In [33]:
# 골든셋: 질문 / 기대 출처(검색이 떠야 함) / 정답에 포함돼야 할 키워드
GOLDEN = [
    {"q":"환불은 며칠 이내에 가능해?",            "evidence":"환불정책", "answer_kw":"14일"},
    {"q":"방문객 주차는 얼마 동안 무료야?",        "evidence":"시설안내", "answer_kw":"2시간"},
    {"q":"점심 식대는 하루 얼마까지 지원돼?",       "evidence":"식대지원", "answer_kw":"1만 2천원"},
    {"q":"재택근무는 일주일에 몇 번 할 수 있어?",   "evidence":"근무정책", "answer_kw":"2회"},
    {"q":"회의실은 최대 몇 분까지 연속 예약할 수 있어?","evidence":"시설안내", "answer_kw":"90분"},
]

def evaluate(vecs, chunks, golden=GOLDEN, k=3, verbose=True):
    r_hit = a_hit = 0
    for g in golden:
        hits = search(g["q"], vecs, chunks, k)
        retrieved_ok = any(g["evidence"] == src for src, _, _ in hits)   # recall@k
        out, _ = answer(g["q"], vecs, chunks, k, verbose=False)
        answer_ok = g["answer_kw"].replace(" ", "") in out.replace(" ", "")
        r_hit += retrieved_ok; a_hit += answer_ok
        if verbose:
            print(f"{'✅' if retrieved_ok else '❌'}검색 {'✅' if answer_ok else '❌'}답변 | {g['q']}  → {out[:45]}")
    n = len(golden)
    print(f"\n📊 검색 recall@{k} = {r_hit}/{n} = {r_hit/n:.0%}   답변 정확도 = {a_hit}/{n} = {a_hit/n:.0%}")
    return r_hit/n, a_hit/n

evaluate(chunk_vecs, chunks)

✅검색 ✅답변 | 환불은 며칠 이내에 가능해?  → 환불은 구매일 기준 14일 안에 접수된 건에 한해 처리됩니다. [출처]
✅검색 ✅답변 | 방문객 주차는 얼마 동안 무료야?  → 방문객 주차는 최초 2시간까지 무료입니다. [출처]
✅검색 ✅답변 | 점심 식대는 하루 얼마까지 지원돼?  → 점심 식대는 하루 1만 2천원까지 지원됩니다. [출처]
✅검색 ✅답변 | 재택근무는 일주일에 몇 번 할 수 있어?  → 재택근무는 주 2회까지 신청할 수 있습니다. [출처]
✅검색 ✅답변 | 회의실은 최대 몇 분까지 연속 예약할 수 있어?  → 회의실은 최대 90분까지 연속 예약할 수 있습니다. [출처]

📊 검색 recall@3 = 5/5 = 100%   답변 정확도 = 5/5 = 100%


(1.0, 1.0)

## STEP 7 — 같은 골든셋 평가를 벡터 DB로

STEP 6와 **같은 골든셋**을 Chroma 검색으로 채점합니다. 점수가 numpy 평가와 같게 나와야 정상 —
**인덱스 구현이 바뀌어도(numpy → Chroma → Pinecone) 골든셋 평가는 그대로 재사용**된다는 것이 평가의 힘입니다.

⌨️ **터미널 Codex:** `codex "ChromaDB 기반 RAG를 같은 골든셋으로 recall@k와 답변 정확도를 채점해줘"`

In [34]:
def evaluate_db(golden=GOLDEN, k=3):
    r_hit = a_hit = 0
    for g in golden:
        hits = search_db(g["q"], k)
        retrieved_ok = any(g["evidence"] == src for src, _, _ in hits)   # recall@k
        out = answer_db(g["q"], k)
        answer_ok = g["answer_kw"].replace(" ", "") in out.replace(" ", "")
        r_hit += retrieved_ok; a_hit += answer_ok
        print(f"{'✅' if retrieved_ok else '❌'}검색 {'✅' if answer_ok else '❌'}답변 | {g['q']}")
    n = len(golden)
    print(f"\n📊 [Chroma] 검색 recall@{k} = {r_hit}/{n} = {r_hit/n:.0%}   답변 정확도 = {a_hit}/{n} = {a_hit/n:.0%}")
    print("👉 numpy 평가(STEP 6)와 같은 점수가 나오면, 인덱스 교체가 검색 품질을 해치지 않았다는 뜻")

evaluate_db()

✅검색 ✅답변 | 환불은 며칠 이내에 가능해?
✅검색 ✅답변 | 방문객 주차는 얼마 동안 무료야?
✅검색 ✅답변 | 점심 식대는 하루 얼마까지 지원돼?
✅검색 ✅답변 | 재택근무는 일주일에 몇 번 할 수 있어?
✅검색 ✅답변 | 회의실은 최대 몇 분까지 연속 예약할 수 있어?

📊 [Chroma] 검색 recall@3 = 5/5 = 100%   답변 정확도 = 5/5 = 100%
👉 numpy 평가(STEP 6)와 같은 점수가 나오면, 인덱스 교체가 검색 품질을 해치지 않았다는 뜻


## STEP 8 — 지표의 사다리: MRR · nDCG

recall@k는 "정답이 top-k에 **들어왔나**"만 봅니다. 강의의 사다리를 한 칸씩 올라가 봅니다:

- **MRR** (Mean Reciprocal Rank·평균 역순위) — 첫 정답이 **몇 등**인가. 1등=1점, 3등=1/3점을 질문 전체로 평균
- **nDCG** (normalized Discounted Cumulative Gain) — 관련 청크가 **위쪽에 몰려 있나**. 순위가 내려갈수록 점수를 할인해서 합산

⌨️ **터미널 Codex:** `codex "골든셋으로 검색 결과의 MRR과 nDCG를 계산하는 함수를 만들어줘"`

In [35]:
def rank_of_first_hit(g, k=10):
    hits = search(g["q"], chunk_vecs, chunks, k)
    for r, (src, _, _) in enumerate(hits, start=1):
        if src == g["evidence"]: return r                 # 첫 정답 청크의 순위
    return None

def mrr(golden=GOLDEN, k=10):
    return sum(1/r for g in golden if (r := rank_of_first_hit(g, k))) / len(golden)

def ndcg(g, k=5):
    hits = search(g["q"], chunk_vecs, chunks, k)
    rels = [1 if src == g["evidence"] else 0 for src, _, _ in hits]   # 관련=1 (이진 관련도)
    dcg  = sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))    # 아래 순위일수록 할인
    n_rel = sum(1 for s, _ in chunks if s == g["evidence"])           # 이상적: 관련 청크가 전부 위에
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, n_rel)))
    return dcg / idcg

print(f"MRR@10 = {mrr():.3f}   (1.0 = 모든 질문에서 정답 문서가 1등)\n")
for g in GOLDEN:
    print(f"  첫 정답 {rank_of_first_hit(g)}위 · nDCG@5 {ndcg(g):.2f} | {g['q']}")
# 질문은 하나다 — '정답 청크가 몇 등으로 나왔나'. recall → MRR → nDCG는 그걸 점점 정밀하게 본다

MRR@10 = 1.000   (1.0 = 모든 질문에서 정답 문서가 1등)

  첫 정답 1위 · nDCG@5 0.77 | 환불은 며칠 이내에 가능해?
  첫 정답 1위 · nDCG@5 1.00 | 방문객 주차는 얼마 동안 무료야?
  첫 정답 1위 · nDCG@5 1.00 | 점심 식대는 하루 얼마까지 지원돼?
  첫 정답 1위 · nDCG@5 0.85 | 재택근무는 일주일에 몇 번 할 수 있어?
  첫 정답 1위 · nDCG@5 0.97 | 회의실은 최대 몇 분까지 연속 예약할 수 있어?


## STEP 9 — UMBRELA: 정답 라벨 없이 LLM이 관련도를 채점

골든셋을 만드는 건 비쌉니다. **UMBRELA**(UMbrela is the Bing RELevance Assessor)의 아이디어 —
정답 라벨 없이 **LLM이 질의↔문서 관련도를 0~3으로 자동 채점**해 평가 데이터 구축 부담을 줄입니다.

In [36]:
def umbrela(query, k=3):
    print(f"질의: {query}")
    for src, c, _ in search(query, chunk_vecs, chunks, k):
        grade = client.chat.completions.create(model=MODEL, temperature=0, messages=[
            {"role":"system","content":"질의와 문서의 관련도를 0~3 정수 하나로만 답하라. 0=무관, 1=주변적, 2=부분 관련, 3=직접 답을 포함"},
            {"role":"user","content":f"질의: {query}\n문서: {c}"}]).choices[0].message.content.strip()
        print(f"  관련도 {grade} | ({src}) {c[:40]}…")

umbrela("환불은 며칠 이내에 가능해?")
# 사람이 만든 라벨(GOLDEN) 없이도 관련도 채점이 나온다 → 대규모 오프라인 평가의 재료

질의: 환불은 며칠 이내에 가능해?
  관련도 3 | (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접수할…
  관련도 3 | (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디지털…
  관련도 2 | (교환반품정책) 개봉하지 않은 상품은 수령일로부터 30일 이내에 교환을 신청할 수 있으며…


## STEP 10 — 🔬 청크 크기 실험 (Exercise)

청크가 **작으면** 사실이 조각나 맥락을 잃고, **크면** 노이즈와 토큰 비용이 커집니다.
이번 문서들은 일부러 **질문의 주어와 정답 숫자를 한 문장 안에서 60자 이상 떨어뜨려** 두었습니다:

- **크기 60** → 주어("방문객 주차는…")와 정답("2시간")이 **다른 청크로 분리** → 검색은 주어 청크만 찾고, 그 안에 답이 없어 **답변 정확도가 떨어진다**
- **크기 120** → 한 청크에 주어+정답이 함께 담김 → 정상
- **크기 300** → 점수는 비슷해도 **컨텍스트(=토큰 비용)가 커지고**, 어휘가 겹치는 방해 문서(교환반품·출장정책)가 통째로 끌려올 위험이 생긴다

⌨️ **터미널 Codex:** `codex "청크 크기를 60/120/300으로 바꿔가며 RAG 평가 점수를 비교하는 실험을 돌려줘"`

In [37]:
def build_and_eval(size, overlap=20, k=3):
    ch = build_chunks(DOCS, size=size, overlap=overlap)
    vecs = build_index(ch)
    print(f"\n=== 청크 크기 {size} (청크 {len(ch)}개) ===")
    return (size,) + build_and_eval_scores(vecs, ch, k)

def build_and_eval_scores(vecs, ch, k):
    r = a = ctx_chars = 0
    for g in GOLDEN:
        hits = search(g["q"], vecs, ch, k)
        ctx_chars += sum(len(c) for _, c, _ in hits)
        if any(g["evidence"] == s for s, _, _ in hits): r += 1
        out, _ = answer(g["q"], vecs, ch, k, verbose=False)
        if g["answer_kw"].replace(" ","") in out.replace(" ",""): a += 1
    n = len(GOLDEN)
    print(f"검색 recall@{k} = {r/n:.0%}   답변 정확도 = {a/n:.0%}   평균 컨텍스트 {ctx_chars//n}자")
    return r/n, a/n, ctx_chars//n

results = [build_and_eval(s) for s in (30, 60, 120, 300)]
print("\n🔬 요약 (크기 → 검색 / 답변 / 평균 컨텍스트):")
for size, rr, aa, cc in results:
    print(f"  크기 {size:>3} → 검색 {rr:.0%}, 답변 {aa:.0%}, 컨텍스트 {cc}자")
# 👉  답변 정확도가 떨어지는 이유: 주어("환불은…")와 정답("14일")이 서로 다른
#    청크로 쪼개져, 검색된 청크 안에 정답이 없다 — 강의의 '잘못된 청킹 = 정반대 답' 그대로.
# 👉 크기 300은 점수가 좋아도 컨텍스트가 커진다 = 요청당 토큰 비용 증가.


=== 청크 크기 30 (청크 97개) ===
검색 recall@3 = 100%   답변 정확도 = 60%   평균 컨텍스트 84자

=== 청크 크기 60 (청크 27개) ===
검색 recall@3 = 100%   답변 정확도 = 80%   평균 컨텍스트 168자

=== 청크 크기 120 (청크 13개) ===
검색 recall@3 = 100%   답변 정확도 = 100%   평균 컨텍스트 292자

=== 청크 크기 300 (청크 6개) ===
검색 recall@3 = 100%   답변 정확도 = 100%   평균 컨텍스트 451자

🔬 요약 (크기 → 검색 / 답변 / 평균 컨텍스트):
  크기  30 → 검색 100%, 답변 60%, 컨텍스트 84자
  크기  60 → 검색 100%, 답변 80%, 컨텍스트 168자
  크기 120 → 검색 100%, 답변 100%, 컨텍스트 292자
  크기 300 → 검색 100%, 답변 100%, 컨텍스트 451자


## STEP 11 — 질의 재작성: '명령'을 빼면 검색이 산다

강의의 질의 재작성 — 명령이 섞인 질의를 그대로 검색하면 벡터가 오염됩니다. 맥락만 추출해 비교해 봅니다.

In [38]:
# '시'와 어울리는 방해 청크를 추가하면, 명령이 섞인 질의의 '벡터 오염'이 순위로 보인다
poem = ("동호회소식",
        "사내 문예 동호회가 봄맞이 시화전을 열었다. 회원들이 직접 쓴 짧은 시 한 편씩을 "
        "로비에 전시하고, 점심시간에는 시 낭송회도 진행한다.")
pv = embed([poem[1]]); pv = pv / np.linalg.norm(pv, axis=1, keepdims=True)
ext_chunks = chunks + [poem]
ext_vecs = np.vstack([chunk_vecs, pv])

raw_q = "우리 환불 규정을 소재로 짧은 시를 한 편 써줘"

def rewrite(q):
    return client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role":"system","content":"질의에서 문서 검색에 쓸 '맥락 키워드'만 추출하라. 명령(시 써줘 등)은 제거. 한 구절만 출력."},
        {"role":"user","content":q}]).choices[0].message.content.strip()

rq = rewrite(raw_q)
print("원 질의   :", raw_q)
print("검색 질의 :", rq)
for label, q in [("그대로 검색", raw_q), ("재작성 검색", rq)]:
    print(f"\n[{label}]")
    for src, c, sc in search(q, ext_vecs, ext_chunks, k=2):
        print(f"  {sc:.3f} ({src}) {c[:38]}…")
# 관찰: 명령("시 써줘")이 섞이면 '동호회소식(시화전)' 청크가 순위를 치고 올라오고
#       환불정책 점수는 뚝 떨어진다. 맥락("환불 규정")만 남기면 환불정책이 압도적 1위.

원 질의   : 우리 환불 규정을 소재로 짧은 시를 한 편 써줘
검색 질의 : 환불 규정

[그대로 검색]
  0.386 (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접…
  0.336 (시설안내) , 최대 90분까지 연속 예약이 가능하다.…

[재작성 검색]
  0.638 (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접…
  0.487 (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디…


## STEP 12 — 하이브리드 검색: 시맨틱 순위 + 키워드 순위 (RRF)

임베딩은 **처음 보는 고유명사(주문번호 등)에 약합니다.** 강의의 하이브리드 슬라이드 그대로 —
두 검색을 **각각의 순위**로 융합하는 **RRF(Reciprocal Rank Fusion)** 를 만들어 봅니다.

> RRF 점수 = 1/(60+시맨틱 순위) + 1/(60+키워드 순위) — 점수 스케일이 달라도 순위만 쓰므로 안전.

In [39]:
# 데모용: 정확 토큰(B73973)만이 단서인 '주문 처리 이력' 청크를 추가
order = ("주문처리이력",
         "접수번호 B73973 건은 5월 2일 오전에 등록되었고, 담당자 확인을 거쳐 5월 4일에 완료로 기록되어 있다.")
ov = embed([order[1]]); ov = ov / np.linalg.norm(ov, axis=1, keepdims=True)
hyb_chunks = chunks + [order]
hyb_vecs = np.vstack([chunk_vecs, ov])

def sem_ranks(q):
    qv = embed([q])[0]; qv = qv / np.linalg.norm(qv)
    sims = hyb_vecs @ qv                                  # 단위 벡터 점곱 = 코사인
    return {int(i): r + 1 for r, i in enumerate(np.argsort(-sims))}, sims

def kw_ranks(q):                                          # BM25의 미니어처: 겹친 토큰 수로 순위
    scores = [len(set(q.split()) & set(c.split())) for _, c in hyb_chunks]
    hit = sorted((i for i, s in enumerate(scores) if s > 0), key=lambda i: -scores[i])
    return {i: r + 1 for r, i in enumerate(hit)}          # 한 토큰도 안 겹치면 순위 없음

def rrf_search(q, k=3, C=60):
    sr, sims = sem_ranks(q); kr = kw_ranks(q)
    rrf = {i: 1/(C+sr[i]) + (1/(C+kr[i]) if i in kr else 0) for i in sr}
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return [(rrf[i], float(sims[i]), hyb_chunks[i]) for i in top]

q = "B73973 주문의 환불은 언제 처리돼?"                    # 고유명사(주문번호)가 섞인 질의
sr, sims = sem_ranks(q)
print(f"'주문처리이력' 청크의 시맨틱 순위: {sr[len(hyb_chunks)-1]}위 / {len(hyb_chunks)}개\n")
print("=== 시맨틱만 (임베딩 top-3) ===")
for i in np.argsort(-sims)[:3]:
    print(f"  cos {sims[i]:.3f}  ({hyb_chunks[i][0]}) {hyb_chunks[i][1][:38]}…")
print("\n=== 하이브리드 (RRF top-3) ===")
for rrf_s, sim, (src, c) in rrf_search(q):
    print(f"  RRF {rrf_s:.4f}  ({src}) {c[:38]}…")
# 관찰: 시맨틱만으로는 '환불' 일반 조항들이 앞서지만, 키워드 검색에서 B73973을
#       정확 매칭한 주문처리이력 청크가 RRF 융합으로 1위에 올라온다 — 강의 도식 그대로.

'주문처리이력' 청크의 시맨틱 순위: 3위 / 15개

=== 시맨틱만 (임베딩 top-3) ===
  cos 0.556  (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디…
  cos 0.530  (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접…
  cos 0.435  (주문처리이력) 접수번호 B73973 건은 5월 2일 오전에 등록되었고, 담당자 확인…

=== 하이브리드 (RRF top-3) ===
  RRF 0.0323  (주문처리이력) 접수번호 B73973 건은 5월 2일 오전에 등록되었고, 담당자 확인…
  RRF 0.0164  (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디…
  RRF 0.0161  (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접…


## STEP 13 — 리랭킹: 2단계 검색 완성 (넓게 후보 → 정예 선별)

강의의 **2단계 검색 파이프라인** — 유사도 top-k를 그대로 LLM에 주면 중복·노이즈가 섞입니다.
**1단계**는 빠르고 넓게(top-6 후보), **2단계**는 리랭커가 질의+청크를 **함께 읽고** 정예만 골라냅니다.

> 실전 리랭커는 cross-encoder 모델(질의·청크를 한 입력으로 넣어 attention으로 비교)을 쓰지만,
> 여기서는 같은 원리를 **LLM 리랭커**로 체험합니다.

⌨️ **터미널 Codex:** `codex "시맨틱 top-6 후보를 LLM 리랭커로 정예 top-2만 추리는 2단계 검색을 만들어줘"`

In [40]:
import re as _re

def rerank(query, candidates, top_n=2):
    numbered = "\n".join(f"[{i}] ({src}) {c}" for i, (src, c, _) in enumerate(candidates))
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role":"system","content":"질문에 답하는 데 실제로 필요한 청크의 번호만 관련도 순으로 골라라. 번호만 쉼표로 출력하라. 예: 2,0"},
        {"role":"user","content":f"질문: {query}\n\n{numbered}"}])
    picks = [int(x) for x in _re.findall(r"\d+", resp.choices[0].message.content)]
    return [candidates[i] for i in picks[:top_n] if i < len(candidates)]

q = "환불은 며칠 이내에 가능해?"
cands = search(q, chunk_vecs, chunks, k=6)                # 1단계: 빠르고 넓게 후보 수집
print("=== 1단계 · 후보 top-6 (시맨틱) — 노이즈가 섞여 있다 ===")
for src, c, sc in cands: print(f"  {sc:.3f} ({src}) {c[:36]}…")

top = rerank(q, cands)                                    # 2단계: 질의+청크를 함께 읽고 선별
print("\n=== 2단계 · 리랭크 결과 (정예 top-2) ===")
for src, c, _ in top: print(f"  ({src}) {c[:36]}…")
# 관찰: 교환반품·출장 같은 어휘만 겹친 후보가 걸러진다.
#       유사도(각자 임베딩)로는 못 하는 판단을, 함께 읽기(cross-encoder 원리)가 해낸다.

=== 1단계 · 후보 top-6 (시맨틱) — 노이즈가 섞여 있다 ===
  0.631 (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를…
  0.533 (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단,…
  0.435 (교환반품정책) 개봉하지 않은 상품은 수령일로부터 30일 이내에 교환을 신청할 수…
  0.410 (시설안내) , 최대 90분까지 연속 예약이 가능하다.…
  0.369 (교환반품정책) 비 없이 처리되며, 영수증 또는 결제 내역이 있어야 접수된다.…
  0.351 (환불정책) 취소된다.…

=== 2단계 · 리랭크 결과 (정예 top-2) ===
  (환불정책) 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를…
  (환불정책)  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단,…


## STEP 14 — 답변 채점의 사다리: ROUGE-L(표면) vs 의미 유사도

강의의 채점 방법 사다리 — 키워드 포함(STEP 6) 다음 칸들을 직접 만듭니다.

- **ROUGE-L**: 정답↔생성 답의 **최장 공통 부분수열(LCS)** 비율. 글자 겹침만 본다
- **BERTScore의 아이디어**: 표면이 달라도 **임베딩(의미)으로 비교**하면 같은 뜻을 잡아낸다

> 단, 의미 유사도도 "문서에 근거했는가"는 못 봅니다 — 그건 다음 STEP의 **Judge**가 합니다.

In [41]:
def lcs_len(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i, ca in enumerate(a):
        for j, cb in enumerate(b):
            dp[i+1][j+1] = dp[i][j]+1 if ca == cb else max(dp[i][j+1], dp[i+1][j])
    return dp[-1][-1]

def rouge_l(ref, hyp):                        # F1 (글자 단위 LCS)
    l = lcs_len(ref, hyp); p, r = l/len(hyp), l/len(ref)
    return 2*p*r/(p+r) if p+r else 0.0

def sem_sim(a, b):                            # BERTScore의 아이디어를 문장 임베딩 코사인으로
    va, vb = embed([a, b])
    return float(va @ vb / (np.linalg.norm(va)*np.linalg.norm(vb)))

ref  = "환불은 구매일로부터 14일 이내에 가능하다"
good = "구매일 기준 2주 안에만 환불할 수 있다"       # 같은 뜻, 다른 표현
bad  = "환불은 구매일로부터 언제든지 가능하다"        # 글자는 비슷, 뜻은 반대

for name, hyp in [("같은 뜻·다른 표현", good), ("비슷한 글자·반대 뜻", bad)]:
    print(f"[{name}] ROUGE-L {rouge_l(ref, hyp):.2f}   의미 유사도 {sem_sim(ref, hyp):.2f} | {hyp}")
# 관찰: ROUGE-L은 '2주'를 오답 취급하고(표면), '언제든지'엔 후하다(글자 겹침).
#       의미 유사도는 바꿔 쓴 표현을 잡지만, 반대 뜻에도 관대할 수 있다 → 최종 판단은 Judge로

[같은 뜻·다른 표현] ROUGE-L 0.36   의미 유사도 0.73 | 구매일 기준 2주 안에만 환불할 수 있다
[비슷한 글자·반대 뜻] ROUGE-L 0.74   의미 유사도 0.84 | 환불은 구매일로부터 언제든지 가능하다


## STEP 15 — LLM-as-a-Judge: Judge LLM에 Rubric 쥐여주기

키워드 포함 여부보다 정교한 채점 — **Judge LLM**이 채점 기준표(**Rubric**)를 보고 **Faithfulness**(답이 문서에만 근거하는가)와 **Relevance**(질문 의도에 맞는가)를 1~5점으로 매기고 이유를 답니다.

> ROUGE-L(글자 겹침)·BERTScore(의미 유사도) 같은 유사도 지표는 "문서에 근거했는가"를 못 봅니다 — 그걸 보는 게 Judge입니다.

> 강의의 **가드레일**(생성 직후 '답이 청크에 충실한가'를 자동 검증)을 실행 가능한 형태로 만든 것이 바로 이 Judge입니다.


In [ ]:
RUBRIC = """아래 <문서>와 <답변>을 보고 두 항목을 1~5로 채점하라.
- Faithfulness: 답변이 문서 내용에만 근거하는가 (지어낸 것 없음=5)
- Relevance: 답변이 질문 의도에 맞는가
형식: Faithfulness: N / Relevance: N / 이유: 한 줄"""

def judge(question, k=2):
    ctx = "\n".join(f"({src}) {c}" for _rrf, _sim, (src, c) in rrf_search(question, k=k))
    ans = client.chat.completions.create(model=MODEL, messages=[
        {"role":"system","content":"아래 문서만 근거로 답하라. 없으면 모른다고 답하라."},
        {"role":"user","content":f"<문서>\n{ctx}\n</문서>\n질문: {question}"}]).choices[0].message.content
    verdict = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role":"system","content":RUBRIC},
        {"role":"user","content":f"<문서>\n{ctx}\n</문서>\n<질문>{question}</질문>\n<답변>{ans}</답변>"}]).choices[0].message.content
    print("답변:", ans[:100], "\n채점:", verdict)

judge("환불은 며칠 안에 신청해야 하나?")
# 주의: 같은 모델 계열로 채점하면 후해진다(self-preference bias) — 실전에선 Judge를 다른 모델로

답변: 환불은 구매일 기준 14일 안에 신청해야 합니다. 
채점: Faithfulness: 5 / Relevance: 5 / 이유: 답변이 문서의 내용을 정확히 반영하고 있으며, 질문에 대한 명확한 답변을 제공하고 있다.


## STEP 16 — 캐싱: 같은 질문에 두 번 돈 쓰지 않기

강의의 **캐싱 4층** 중 두 층을 직접 만듭니다 — 캐시는 위층에서 맞으면 아래 연산을 전부 생략합니다.

- **응답 캐시**: 같은(정규화된) 질문이 또 오면 검색·LLM **0회**, 저장된 답 즉시 반환
- **임베딩 캐시**: 안 바뀐 텍스트는 다시 임베딩하지 않음 — 문서 100개 중 1개만 수정돼도 99개 절약

⌨️ **터미널 Codex:** `codex "RAG에 응답 캐시와 임베딩 캐시를 dict로 추가하고 두 번째 호출이 빨라지는 걸 보여줘"`

In [ ]:
import time

EMB_CACHE, ANS_CACHE = {}, {}

def cached_embed(texts):
    misses = [t for t in texts if t not in EMB_CACHE]
    if misses:                                            # 캐시에 없는 텍스트만 API 호출
        for t, v in zip(misses, embed(misses)): EMB_CACHE[t] = v
    print(f"  (임베딩 {len(texts)}건 중 API 호출 {len(misses)}건)")
    return np.array([EMB_CACHE[t] for t in texts])

def cached_answer(query, k=3):
    key = " ".join(query.split())                         # 정규화된 질의 = 캐시 키
    if key in ANS_CACHE:
        return ANS_CACHE[key] + "  ⚡(응답 캐시 HIT — 검색·LLM 호출 0회)"
    out, _ = answer(query, chunk_vecs, chunks, k)
    ANS_CACHE[key] = out
    return out

for run in (1, 2):                                        # 같은 질문 두 번
    t0 = time.time()
    out = cached_answer("환불은 며칠 이내에 가능해?")
    print(f"{run}회차 {time.time()-t0:.2f}s | {out[:60]}…")

print("\n[임베딩 캐시] 전체 재인덱싱을 두 번 돌려보면 —")
t0 = time.time(); cached_embed([c for _, c in chunks]); print(f"  1회차 {time.time()-t0:.2f}s (전부 MISS)")
t0 = time.time(); cached_embed([c for _, c in chunks]); print(f"  2회차 {time.time()-t0:.2f}s (전부 HIT)")

📚 검색된 청크:
  (0.631) [환불정책] 환불을 원하는 고객은 구매 채널이나 결제 수단에 관계없이 신청서를 접...
  (0.533) [환불정책]  뒤, 구매일 기준 14일 안에 접수된 건에 한해 처리된다. 단, 디...
  (0.435) [교환반품정책] 개봉하지 않은 상품은 수령일로부터 30일 이내에 교환을 신청할 수 있...

🤖 답: 환불은 구매일 기준 14일 안에 접수된 건에 한해 가능하다. [출처]
1회차 1.49s | 환불은 구매일 기준 14일 안에 접수된 건에 한해 가능하다. [출처]…
2회차 0.00s | 환불은 구매일 기준 14일 안에 접수된 건에 한해 가능하다. [출처]  ⚡(응답 캐시 HIT — 검색·LLM…

[임베딩 캐시] 전체 재인덱싱을 두 번 돌려보면 —
  (임베딩 14건 중 API 호출 14건)
  1회차 0.72s (전부 MISS)
  (임베딩 14건 중 API 호출 0건)
  2회차 0.00s (전부 HIT)


## STEP 17 — 규모의 3차원: 내 MVP 비용 어림 계산기

강의의 스케일링 — 규모는 **문서 양 × 문서 복잡도 × 쿼리 부하** 세 방향으로 커지고,
진짜 비용은 **LLM 입력 토큰**입니다. 숫자를 내 MVP 값으로 바꿔 어림해 보세요.

In [ ]:
# 숫자를 내 MVP 값으로 바꿔보세요 (기본값 = 강의 예시 규모)
docs, chunks_per_doc = 2_000_000, 1            # 문서 양 × 문서 복잡도(문서당 청크)
queries_per_month    = 150_000                 # 쿼리 부하

n_chunks   = docs * chunks_per_doc
storage_gb = n_chunks * 4 / 1024**2                          # 청크당 ≈4KB (1024차원)
embed_usd  = n_chunks * 500 * 0.13 / 1e6                     # 청크당 ~500토큰 × $0.13/1M (embedding-3-large)
query_usd  = queries_per_month * 3000 * 0.15 / 1e6           # 요청당 입력 ~3K토큰 × $0.15/1M (gpt-4o-mini)

print(f"청크 {n_chunks:,}개 → 벡터 저장 ≈ {storage_gb:,.1f} GB")
print(f"초기 임베딩(1회) ≈ ${embed_usd:,.0f}")
print(f"월 질의 LLM 입력 ≈ ${query_usd:,.0f}   ← 캐싱(STEP 16)·모델 라우팅이 깎는 비용")
# 벡터 크기는 청크 길이와 무관하게 고정 → 저장·비용을 '미리' 산정할 수 있다 

청크 2,000,000개 → 벡터 저장 ≈ 7.6 GB
초기 임베딩(1회) ≈ $130
월 질의 LLM 입력 ≈ $68   ← 캐싱(STEP 16)·모델 라우팅이 깎는 비용


## STEP 18 — 실패 지도: 쓰레기가 들어가면 쓰레기가 나온다 (수집 실패 체험)

답이 이상할 때 **실패 지도**는 위에서 아래로 진단하라고 했습니다:
**① 수집**(청크를 눈으로 확인) → **② 검색**(recall@k) → **③ 생성**(Faithfulness).

①을 체험합니다 — 파싱이 망가진 문서(HTML 잡음 포함)를 그대로 인덱싱하면 무슨 일이 생길까요?

In [45]:
html = ("<div class='nav'>홈 | 로그인 | 마이페이지 | 고객센터</div>"
        "<footer>Copyright 2026 MyMVP Inc. 개인정보처리방침 · 이용약관</footer>"
        "배송은 결제 완료 후 영업일 기준 2일 안에 출고된다."
        "<div class='banner'>지금 가입하면 첫 구매 10% 할인! 이벤트 배너</div>")
clean = "배송은 결제 완료 후 영업일 기준 2일 안에 출고된다."   # 파싱 = 본문만 남기기

q = "배송은 며칠 안에 출고돼?"
qv = embed([q])[0]; qv = qv / np.linalg.norm(qv)

for label, text in [("① 파싱 실패 — HTML 그대로", html), ("② 파싱 성공 — 본문만", clean)]:
    ch = chunk_text(text, size=80, overlap=20)
    vecs = embed(ch); vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    i = int(np.argmax(vecs @ qv))
    print(f"[{label}] 청크 {len(ch)}개, top-1 유사도 {float(vecs[i] @ qv):.3f}")
    print(f"   top-1 청크: {ch[i][:55]}…\n")
# 관찰: 오염 인덱스는 잡음 청크가 늘고 유사도가 낮아진다 — 모델·검색을 고치기 전에
#       ① 수집(청크 눈으로 보기)부터. 원천이 깨끗한데도 틀리면 그때 ② 검색 → ③ 생성 순서로

[① 파싱 실패 — HTML 그대로] 청크 4개, top-1 유사도 0.587
   top-1 청크: 준 2일 안에 출고된다.<div class='banner'>지금 가입하면 첫 구매 10% 할인! 이…

[② 파싱 성공 — 본문만] 청크 1개, top-1 유사도 0.694
   top-1 청크: 배송은 결제 완료 후 영업일 기준 2일 안에 출고된다.…



## STEP 19 — ⭐ 내 팀 MVP 문서에 적용

위 파이프라인(청킹 → 인덱싱 → 검색 → 생성 → 평가)은 **그대로** 당신 팀 MVP에 옮길 수 있습니다.
`DOCS`를 우리 서비스 문서로, `GOLDEN`을 우리 질문/정답으로 바꾸면 **내 MVP 전용 RAG + 평가**가 완성됩니다.

⌨️ **터미널 Codex:** `codex "우리 MVP 문서로 DOCS와 골든셋을 채우고 청킹·검색·평가 파이프라인을 다시 돌려줘"`

In [46]:
# 팀별 작성: 우리 MVP의 문서와 골든셋을 채워보세요
MY_DOCS = [
    # ("문서제목", "여러 사실이 섞인 긴 본문 ..."),
    # ("문서제목", "..."),
]
MY_GOLDEN = [
    # {"q":"우리 서비스 질문", "evidence":"문서제목", "answer_kw":"정답키워드"},
]

if MY_DOCS and MY_GOLDEN:
    my_chunks = build_chunks(MY_DOCS, size=120, overlap=30)
    my_vecs = build_index(my_chunks)
    evaluate(my_vecs, my_chunks, golden=MY_GOLDEN)
else:
    print("⬜ MY_DOCS와 MY_GOLDEN을 채운 뒤 다시 실행하세요 → '내 MVP RAG' 완성")

⬜ MY_DOCS와 MY_GOLDEN을 채운 뒤 다시 실행하세요 → '내 MVP RAG' 완성


## 정리

- **RAG = 수집(한 번) + 질의(매번).** 긴 문서는 **청킹**해야 비용·검색 정확도가 산다 — 청킹은 문자열 분할이 아니라 의미 단위 분할.
- 임베딩은 **단위 벡터** → 유사도는 **점곱 = 코사인** 한 번. 검색된 청크**만** 근거로 답해 환각을 줄이고 **출처** 를 제시.
- 인덱스는 numpy로 개념을 익히고 **벡터 DB(Chroma)** 로 옮겼다 — 실전에선 Pinecone·Weaviate·Qdrant·pgvector 중 운영 방식에 맞게 선택.
- **평가 없이는 개선 없다.** 골든셋으로 **검색 품질(recall@k)** 과 **답변 품질** 을 따로 측정하고, **검색기를 먼저** 최적화.
- 검색은 **2단계**(넓게 후보 수집 → 리랭크로 정예 선별)로, 반복 질문·재임베딩은 **캐싱**으로 — 정확도와 비용을 함께 잡는다.
- 청크 크기 실험에서 봤듯 **작은 결정 하나가 품질을 좌우** 한다 — 느낌이 아니라 숫자로.